In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import copy

from data_loader import build_complete_dataset
from window import create_record_windows

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix,
                             roc_curve)
from sklearn.model_selection import ParameterGrid

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\wikma\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\wikma\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\wikma\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\wikma\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\wikma\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\wikma\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\wikma\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\wikma\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:

def extract_window_features(window_df, fs, prefix=""):
    features = {}
    columns = [column for column in window_df.columns if column != "Time"]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
        signal = data[:, i]
        features[f"{prefix}_{col}_mean"] = np.mean(signal)
        features[f"{prefix}_{col}_std"] = np.std(signal)
        features[f"{prefix}_{col}_rms"] = np.sqrt(np.mean(signal ** 2))
        features[f"{prefix}_{col}_max"] = np.max(signal)
        features[f"{prefix}_{col}_min"] = np.min(signal)
        features[f"{prefix}_{col}_ptp"] = np.ptp(signal)
        features[f"{prefix}_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
        features[f"{prefix}_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
        fft_vals = np.abs(np.fft.rfft(signal))
        freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
        features[f"{prefix}_{col}_fft_energy"] = np.sum(fft_vals ** 2)
        features[f"{prefix}_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features


In [ ]:

def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)

    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))

    for i in range(min_len):
        features = {}
        features.update(extract_window_features(acc_windows[i], fs=6700, prefix="acc"))
        features.update(extract_window_features(gyro_windows[i], fs=6700, prefix="gyro"))
        features.update(extract_window_features(mic_windows[i], fs=16000, prefix="mic"))

        features["segment_id"] = record.metadata.get("segment_id")
        features["split_label"] = record.metadata.get("split_label")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)

    return pd.DataFrame(record_features)


In [ ]:

def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)

In [ ]:
path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

if train_path.exists() and test_path.exists():
    X_train_df = pd.read_csv(train_path)
    X_test_df = pd.read_csv(test_path)
else:
    df = build_complete_dataset()
    X_train_df = build_feature_dataset(df[0], create_record_windows)
    X_test_df = build_feature_dataset(df[1], create_record_windows)
    X_train_df.to_csv(train_path, index=False)
    X_test_df.to_csv(test_path, index=False)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

print(X_train_df.columns.tolist())
print(X_test_df.columns.tolist())

X_train_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_train_target = X_train_df[X_train_df["split_label"] == "Normal_Target_Train"].copy()

X_val = X_test_df[
    X_test_df["split_label"].isin(["Normal_Source_Test", "Anomaly_Source_Test"])].copy()  # zbior walidacyjny

X_test = X_test_df[
    X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy()  # to nasz zbiór testowy
metadata_cols = ["segment_id", "anomaly_label", "domain_shift_op", "domain_shift_env"]

acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]
X_train_acc = X_train_df[acc_features].fillna(0)
X_test_acc = X_test[acc_features].fillna(0)

X_train_mic = X_train_df[mic_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)

X_train_gyro = X_train_df[gyro_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)

X_train_source_acc = X_train_source[acc_features].fillna(0)
X_train_source_mic = X_train_source[mic_features].fillna(0)
X_train_source_gyro = X_train_source[gyro_features].fillna(0)

X_train_target_acc = X_train_target[acc_features].fillna(0)
X_train_target_mic = X_train_target[mic_features].fillna(0)
X_train_target_gyro = X_train_target[gyro_features].fillna(0)

X_val_acc = X_val[acc_features].fillna(0)
X_val_mic = X_val[mic_features].fillna(0)
X_val_gyro = X_val[gyro_features].fillna(0)

X_test_acc = X_test[acc_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)


Train: (102984, 75)
Test : (25984, 75)
['acc_A_x [g]_mean', 'acc_A_x [g]_std', 'acc_A_x [g]_rms', 'acc_A_x [g]_max', 'acc_A_x [g]_min', 'acc_A_x [g]_ptp', 'acc_A_x [g]_skew', 'acc_A_x [g]_kurtosis', 'acc_A_x [g]_fft_energy', 'acc_A_x [g]_fft_peak_freq', 'acc_A_y [g]_mean', 'acc_A_y [g]_std', 'acc_A_y [g]_rms', 'acc_A_y [g]_max', 'acc_A_y [g]_min', 'acc_A_y [g]_ptp', 'acc_A_y [g]_skew', 'acc_A_y [g]_kurtosis', 'acc_A_y [g]_fft_energy', 'acc_A_y [g]_fft_peak_freq', 'acc_A_z [g]_mean', 'acc_A_z [g]_std', 'acc_A_z [g]_rms', 'acc_A_z [g]_max', 'acc_A_z [g]_min', 'acc_A_z [g]_ptp', 'acc_A_z [g]_skew', 'acc_A_z [g]_kurtosis', 'acc_A_z [g]_fft_energy', 'acc_A_z [g]_fft_peak_freq', 'gyro_G_x [mdps]_mean', 'gyro_G_x [mdps]_std', 'gyro_G_x [mdps]_rms', 'gyro_G_x [mdps]_max', 'gyro_G_x [mdps]_min', 'gyro_G_x [mdps]_ptp', 'gyro_G_x [mdps]_skew', 'gyro_G_x [mdps]_kurtosis', 'gyro_G_x [mdps]_fft_energy', 'gyro_G_x [mdps]_fft_peak_freq', 'gyro_G_y [mdps]_mean', 'gyro_G_y [mdps]_std', 'gyro_G_y [mdps]_

In [ ]:
def find_best_threshold(y_true, scores):
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)
    thresholds = np.percentile(scores, np.linspace(0, 100, 200))
    best_threshold = thresholds[0]
    best_f1 = -1
    for threshold in thresholds:
        y_pred = (scores >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, num_layers=1):
        super().__init__()
        self.encoder_lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2 if num_layers>1 else 0)
        self.encoder_fc = nn.Linear(hidden_dim, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, hidden_dim)
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2 if num_layers>1 else 0)
        self.output_fc = nn.Linear(hidden_dim, input_dim)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, cell) = self.encoder_lstm(x)
        latent = self.encoder_fc(hidden[-1])
        decoded = torch.tanh(self.decoder_fc(latent)).unsqueeze(1).repeat(1, seq_len, 1)
        h0 = decoded.mean(dim=1).unsqueeze(0).repeat(self.num_layers, 1, 1).contiguous()
        c0 = torch.zeros_like(h0)
        output, _ = self.decoder_lstm(decoded, (h0, c0))
        output = self.output_fc(output)
        return output, latent

In [ ]:
def train_pytorch_model(model, X_train_scaled, epochs, batch_size, patience=3):
    # Generowanie zbioru walidacyjnego (10% danych) do Early Stopping
    val_size = int(0.1 * len(X_train_scaled))
    train_size = len(X_train_scaled) - val_size
    indices = np.random.permutation(len(X_train_scaled))

    # Przetworzenie danych na tensory z dodaniem wymiaru sekwencji: [N, 1, input_dim]
    X_tr = torch.tensor(X_train_scaled[indices[:train_size]], dtype=torch.float32).unsqueeze(1)
    X_va = torch.tensor(X_train_scaled[indices[train_size:]], dtype=torch.float32).unsqueeze(1)

    train_loader = DataLoader(TensorDataset(X_tr, X_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_va, X_va), batch_size=batch_size, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        for batch_x, _ in train_loader:
            batch_x = batch_x.to(device)
            optimizer.zero_grad()
            output, _ = model(batch_x)
            loss = criterion(output, batch_x)
            loss.backward()
            optimizer.step()

        # Walidacja
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, _ in val_loader:
                batch_x = batch_x.to(device)
                output, _ = model(batch_x)
                loss = criterion(output, batch_x)
                val_loss += loss.item() * batch_x.size(0)
        val_loss /= len(X_va)

        # Early Stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_weights)
    return model


In [ ]:

def predict_pytorch_model(model, X_scaled, batch_size=64):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_t = torch.tensor(X_scaled, dtype=torch.float32).unsqueeze(1)
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=False)

    model.to(device)
    model.eval()

    predictions = []
    with torch.no_grad():
        for batch_x, in loader:
            batch_x = batch_x.to(device)
            output, _ = model(batch_x)
            predictions.append(output.cpu().numpy())

    # Łączymy wyniki i usuwamy wymiar sekwencyjny [N, 1, input_dim] -> [N, input_dim]
    return np.concatenate(predictions, axis=0).squeeze(1)

In [ ]:


def tune_autoencoder(
        X_train,
        X_val,
        train_df,
        val_df,
        sensor_type,
        param_grid):
    results = []
    y_val = (val_df.groupby("segment_id")["anomaly_label"].first().eq("loosescrewsA").astype(int))

    for params in ParameterGrid(param_grid):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # Inicjalizacja modelu PyTorch
        model = LSTMAutoencoder(
            input_dim=X_train_scaled.shape[1],
            latent_dim=params["encoding_dim"], 
            hidden_dim=64,  
            num_layers=1
        )

        model = train_pytorch_model(
            model,
            X_train_scaled,
            epochs=params["epochs"],
            batch_size=params["batch_size"]
        )


        X_val_pred = predict_pytorch_model(model, X_val_scaled, batch_size=params["batch_size"])

        val_scores = np.mean(np.power(X_val_scaled - X_val_pred, 2), axis=1)

        val_results = pd.DataFrame({"segment_id": val_df["segment_id"].values, "score": val_scores})
        segment_scores = (val_results.groupby("segment_id").agg(score=("score", "max")).reset_index())
        segment_scores = segment_scores.merge(y_val.rename("y_true"), left_on="segment_id", right_index=True,
                                              how="inner")

        threshold, best_f1 = find_best_threshold(segment_scores["y_true"].values, segment_scores["score"].values)
        y_pred = (segment_scores["score"] >= threshold).astype(int)

        precision = precision_score(segment_scores["y_true"], y_pred, zero_division=0)
        recall = recall_score(segment_scores["y_true"], y_pred, zero_division=0)

        accuracy = accuracy_score(segment_scores["y_true"], y_pred)
        roc_auc = roc_auc_score(segment_scores["y_true"], segment_scores["score"])

        results.append({"sensor": sensor_type,
                        **params,
                        "threshold": threshold,
                        "Accuracy": accuracy,
                        "Precision": precision,
                        "Recall": recall,
                        "ROC AUC": roc_auc,
                        "F1": best_f1})

    results_df = pd.DataFrame(results)
    best_row = results_df.loc[results_df["F1"].idxmax()]

    best_params = {
        "encoding_dim": int(best_row["encoding_dim"]),
        "epochs": int(best_row["epochs"]),
        "batch_size": int(best_row["batch_size"])
    }

    best_threshold = best_row["threshold"]
    print(f"Sensor: {sensor_type}")
    print("Parameters:", best_params)
    print(f"Threshold : {best_threshold:.6f}")
    print(f"Accuracy  : {best_row['Accuracy']:.4f}")
    print(f"Precision : {best_row['Precision']:.4f}")
    print(f"Recall    : {best_row['Recall']:.4f}")
    print(f"ROC AUC   : {best_row['ROC AUC']:.4f}")
    print(f"F1        : {best_row['F1']:.4f}\n")

    return best_params, best_threshold, results_df


In [ ]:

param_grid = {
    "encoding_dim": [8, 16, 32],
    "epochs": [30, 50],
    "batch_size": [32, 64]
}

In [ ]:
best_params_acc, threshold_acc, tuning_acc = tune_autoencoder(
    X_train_source_acc,
    X_val_acc,
    X_train_source,
    X_val,
    "ACC source",
    param_grid
)

Sensor: ACC source
Parameters: {'encoding_dim': 16, 'epochs': 30, 'batch_size': 64}
Threshold : 0.004004
Accuracy  : 0.8319
Precision : 0.7939
Recall    : 0.8966
ROC AUC   : 0.8669
F1        : 0.8421



In [ ]:
best_params_mic, threshold_mic, tuning_mic = tune_autoencoder(
    X_train_source_mic,
    X_val_mic,
    X_train_source,
    X_val,
    "MIC source",
    param_grid
)

Sensor: MIC source
Parameters: {'encoding_dim': 8, 'epochs': 30, 'batch_size': 64}
Threshold : 0.000216
Accuracy  : 0.5302
Precision : 0.5156
Recall    : 1.0000
ROC AUC   : 0.5835
F1        : 0.6804



In [ ]:
best_params_gyro, threshold_gyro, tuning_gyro = tune_autoencoder(
    X_train_source_gyro,
    X_val_gyro,
    X_train_source,
    X_val,
    "GYRO source",
    param_grid
)

Sensor: GYRO source
Parameters: {'encoding_dim': 8, 'epochs': 50, 'batch_size': 64}
Threshold : 0.076467
Accuracy  : 0.7931
Precision : 0.7394
Recall    : 0.9052
ROC AUC   : 0.8112
F1        : 0.8140



In [ ]:
def run_autoencoder(X_train, X_test, test_df, sensor_type, best_params, threshold, X_train_target=None, threshold_method="percentile_99"):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Inicjalizacja modelu PyTorch
    model = LSTMAutoencoder(
        input_dim=X_train_scaled.shape[1],
        latent_dim=best_params["encoding_dim"],
        hidden_dim=64,
        num_layers=1
    )

    model = train_pytorch_model(
        model,
        X_train_scaled,
        epochs=best_params["epochs"],
        batch_size=best_params["batch_size"]
    )

    if X_train_target is not None and not X_train_target.empty:
        X_tgt_scaled = scaler.transform(X_train_target)
        tgt_pred = predict_pytorch_model(model, X_tgt_scaled, batch_size=best_params["batch_size"])
        tgt_mse = np.mean(np.power(X_tgt_scaled - tgt_pred, 2), axis=1)

        if threshold_method == "percentile_99":
            used_threshold = np.percentile(tgt_mse, 99.5) 
        elif threshold_method == "3_sigma":
            used_threshold = np.mean(tgt_mse) + 3 * np.std(tgt_mse)
        else:
            used_threshold = np.percentile(tgt_mse, 95)
    else:
        used_threshold = threshold

    X_test_pred = predict_pytorch_model(model, X_test_scaled, batch_size=best_params["batch_size"])
    test_scores = np.mean(np.power(X_test_scaled - X_test_pred, 2), axis=1)

    results = pd.DataFrame({"segment_id": test_df["segment_id"].values,
                            "anomaly_label": test_df["anomaly_label"].values,
                            "score": test_scores})

    segment_scores = (results.groupby("segment_id").agg(score=("score", "max"),
                                                        anomaly_label=("anomaly_label", "first")).reset_index())
    segment_scores["y_true"] = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)
    segment_scores["y_pred"] = (segment_scores["score"] >= used_threshold).astype(int)

    y_true = segment_scores["y_true"]
    y_pred = segment_scores["y_pred"]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, segment_scores["score"])
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"{sensor_type} | Method: {threshold_method}")
    print(f"Threshold: {used_threshold:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(cm)
    print("-" * 30)

    metrics = {"Sensors": sensor_type, "Accuracy": accuracy, "Precision": precision,
               "Recall": recall, "F1": f1, "ROC_AUC": roc_auc,
               "TN": cm[0, 0], "FP": cm[0, 1], "FN": cm[1, 0], "TP": cm[1, 1]}

    return metrics, segment_scores, model

Trenowanie
i
testowanie
osobnych
modeli(Autoenkodery)
wyłącznie
na
domenie
source

In [ ]:

metrics_acc_source, pred_acc_source, model_acc_source = run_autoencoder(
    X_train_source_acc, X_val_acc, X_val, "ACC - source", best_params_acc, threshold_acc
)
metrics_mic_source, pred_mic_source, model_mic_source = run_autoencoder(
    X_train_source_mic, X_val_mic, X_val, "MIC - source", best_params_mic, threshold_mic
)
metrics_gyro_source, pred_gyro_source, model_gyro_source = run_autoencoder(
    X_train_source_gyro, X_val_gyro, X_val, "GYRO - source", best_params_gyro, threshold_gyro
)

ACC - source
Threshold: 0.0040
Accuracy : 0.6767
Precision: 0.6145
Recall   : 0.9483
F1-score : 0.7458
ROC AUC  : 0.8000
[[ 47  69]
 [  6 110]]
MIC - source
Threshold: 0.0002
Accuracy : 0.5129
Precision: 0.5067
Recall   : 0.9828
F1-score : 0.6686
ROC AUC  : 0.5751
[[  5 111]
 [  2 114]]
GYRO - source
Threshold: 0.0765
Accuracy : 0.6897
Precision: 0.6279
Recall   : 0.9310
F1-score : 0.7500
ROC AUC  : 0.7277
[[ 52  64]
 [  8 108]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię(Source -> Source)

In [ ]:
final_source_results = (pred_acc_source[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                        .merge(pred_mic_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}),
                               on="segment_id")
                        .merge(pred_gyro_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}),
                               on="segment_id"))

final_source_results["y_pred_final"] = (
        final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)
y_true_source = final_source_results["y_true"]
y_pred_source_final = final_source_results["y_pred_final"]

print("Final Source Results (at least 1 sensor):")
print(f"Accuracy : {accuracy_score(y_true_source, y_pred_source_final):.4f}")
print(f"Precision: {precision_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1]))

Final Source Results (at least 1 sensor):
Accuracy : 0.5129
Precision: 0.5066
Recall   : 1.0000
F1-score : 0.6725
Confusion matrix:
 [[  3 113]
 [  0 116]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię(Source -> Source)

In [ ]:
final_source_results["y_pred_final"] = (
        final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_pred_source_final = final_source_results["y_pred_final"]

print("Final Source Results (majority voting):")
print(f"Accuracy : {accuracy_score(y_true_source, y_pred_source_final):.4f}")
print(f"Precision: {precision_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1]))

Final Source Results (majority voting):
Accuracy : 0.6078
Precision: 0.5622
Recall   : 0.9741
F1-score : 0.7129
Confusion matrix:
 [[ 28  88]
 [  3 113]]


In [ ]:
def evaluate_acc_gyro_only(pred_acc, pred_gyro, title):
    """
    Funkcja łączy predykcje z akcelerometru i żyroskopu, ignorując mikrofon.
    Segment jest anomalią tylko, gdy oba czujniki (ACC i GYRO) wskażą anomalię.
    """
    # Łączenie wyników tylko dla ACC i GYRO
    df_combined = (pred_acc[["segment_id", "y_true", "y_pred"]]
                   .rename(columns={"y_pred": "pred_acc"})
                   .merge(pred_gyro[["segment_id", "y_pred"]]
                          .rename(columns={"y_pred": "pred_gyro"}),
                          on="segment_id"))

    # Uznajemy za anomalię, jeśli oba modele wskażą 1
    df_combined["y_pred_final"] = ((df_combined["pred_acc"] == 1) & (df_combined["pred_gyro"] == 1)).astype(int)

    y_true = df_combined["y_true"]
    y_pred = df_combined["y_pred_final"]

    print(f"--- {title} ---")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred, labels=[0, 1]))

    return df_combined

In [ ]:
final_acc_gyro_source = evaluate_acc_gyro_only(
    pred_acc_source,
    pred_gyro_source,
    "Source -> Source (ACC & GYRO Only)"
)

--- Source -> Source (ACC & GYRO Only) ---
Accuracy : 0.7543
Precision: 0.6980
Recall   : 0.8966
F1-score : 0.7849
Confusion matrix:
 [[ 71  45]
 [ 12 104]]


Trenowanie
osobnych
modeli
dla
każdego
czujnika
wyłącznie
z
wykorzystaniem
domeny
source, a
testowanie
na
target

In [ ]:
metrics_acc_tgt, pred_acc_tgt, model_acc_tgt = run_autoencoder(
    X_train_source_acc, X_test_acc, X_test, "ACC source -> target", best_params_acc, threshold_acc,X_train_target=X_train_target_acc
)
metrics_mic_tgt, pred_mic_tgt, model_mic_tgt = run_autoencoder(
    X_train_source_mic, X_test_mic, X_test, "MIC source -> target", best_params_mic, threshold_mic,
    X_train_target=X_train_target_mic
)
metrics_gyro_tgt, pred_gyro_tgt, model_gyro_tgt = run_autoencoder(
    X_train_source_gyro, X_test_gyro, X_test, "GYRO source -> target", best_params_gyro, threshold_gyro,
    X_train_target=X_train_target_gyro
)


ACC source -> target
Threshold: 0.0104
Accuracy : 0.3836
Precision: 0.3689
Recall   : 0.3276
F1-score : 0.3470
ROC AUC  : 0.3908
[[51 65]
 [78 38]]
MIC source -> target
Threshold: 0.0014
Accuracy : 0.5000
Precision: 0.5000
Recall   : 0.7155
F1-score : 0.5887
ROC AUC  : 0.5114
[[33 83]
 [33 83]]
GYRO source -> target
Threshold: 0.1442
Accuracy : 0.4224
Precision: 0.4357
Recall   : 0.5259
F1-score : 0.4766
ROC AUC  : 0.3963
[[37 79]
 [55 61]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię(Source -> Target)

In [ ]:
final_tgt_results = (pred_acc_tgt[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                     .merge(pred_mic_tgt[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}),
                            on="segment_id")
                     .merge(pred_gyro_tgt[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}),
                            on="segment_id"))

final_tgt_results["y_pred_final"] = (final_tgt_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(
    int)
y_true_tgt = final_tgt_results["y_true"]
y_pred_tgt_final = final_tgt_results["y_pred_final"]

print("Final Results source -> target (at least 1 sensor):")
print(f"Accuracy : {accuracy_score(y_true_tgt, y_pred_tgt_final):.4f}")
print(f"Precision: {precision_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_tgt, y_pred_tgt_final, labels=[0, 1]))


Final Results source -> target (at least 1 sensor):
Accuracy : 0.4914
Precision: 0.4954
Recall   : 0.9310
F1-score : 0.6467
Confusion matrix:
 [[  6 110]
 [  8 108]]


Teraz
segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię(Source -> Target)

In [ ]:

final_tgt_results["y_pred_final"] = (final_tgt_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(
    int)
y_pred_tgt_final = final_tgt_results["y_pred_final"]

print("Final Results source -> target (majority voting):")
print(f"Accuracy : {accuracy_score(y_true_tgt, y_pred_tgt_final):.4f}")
print(f"Precision: {precision_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_tgt, y_pred_tgt_final, labels=[0, 1]))


Final Results source -> target (majority voting):
Accuracy : 0.3966
Precision: 0.4178
Recall   : 0.5259
F1-score : 0.4656
Confusion matrix:
 [[31 85]
 [55 61]]


In [ ]:
final_acc_gyro_tgt = evaluate_acc_gyro_only(
    pred_acc_tgt,
    pred_gyro_tgt,
    "Source -> Target (ACC & GYRO Only)"
)

--- Source -> Target (ACC & GYRO Only) ---
Accuracy : 0.3879
Precision: 0.3243
Recall   : 0.2069
F1-score : 0.2526
Confusion matrix:
 [[66 50]
 [92 24]]


Trenowanie
i
testowanie
osobnych
modeli
dla
każdego
czujnika
z
wykorzystaniem
całych
zbiorów
treningowego
i
testowego

In [ ]:
X_test_full_acc = X_test_df[acc_features].fillna(0)
X_test_full_mic = X_test_df[mic_features].fillna(0)
X_test_full_gyro = X_test_df[gyro_features].fillna(0)

In [ ]:
metrics_acc_all, pred_acc_all, model_acc_all = run_autoencoder(
    X_train_acc, X_test_full_acc, X_test_df, "ACC (ALL)",
    best_params_acc, threshold_acc,
    X_train_target=X_train_acc
)
metrics_mic_all, pred_mic_all, model_mic_all = run_autoencoder(
    X_train_mic, X_test_full_mic, X_test_df, "MIC (ALL)",
    best_params_mic, threshold_mic,
    X_train_target=X_train_mic
)
metrics_gyro_all, pred_gyro_all, model_gyro_all = run_autoencoder(
    X_train_gyro, X_test_full_gyro, X_test_df, "GYRO (ALL)",
    best_params_gyro, threshold_gyro,
    X_train_target=X_train_gyro
)

ACC (ALL) | Method: percentile_99
Threshold: 0.0051
Accuracy : 0.6228
Precision: 0.5979
Recall   : 0.7500
F1-score : 0.6654
[[115 117]
 [ 58 174]]
------------------------------
MIC (ALL) | Method: percentile_99
Threshold: 0.0579
Accuracy : 0.5086
Precision: 0.5500
Recall   : 0.0948
F1-score : 0.1618
[[214  18]
 [210  22]]
------------------------------
GYRO (ALL) | Method: percentile_99
Threshold: 0.1210
Accuracy : 0.5582
Precision: 0.5529
Recall   : 0.6078
F1-score : 0.5791
[[118 114]
 [ 91 141]]
------------------------------


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię(ALL -> ALL)

In [ ]:
final_all_results = (pred_acc_all[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                     .merge(pred_mic_all[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}),
                            on="segment_id")
                     .merge(pred_gyro_all[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}),
                            on="segment_id"))

final_all_results["y_pred_final"] = (final_all_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(
    int)
y_true_all = final_all_results["y_true"]
y_pred_all_final = final_all_results["y_pred_final"]

print("Final Results ALL -> ALL (at least 1 sensor):")
print(f"Accuracy : {accuracy_score(y_true_all, y_pred_all_final):.4f}")
print(f"Precision: {precision_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_all, y_pred_all_final, labels=[0, 1]))

Final Results ALL -> ALL (at least 1 sensor):
Accuracy : 0.5948
Precision: 0.5601
Recall   : 0.8836
F1-score : 0.6856
Confusion matrix:
 [[ 71 161]
 [ 27 205]]


Teraz
segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię(ALL -> ALL)

In [ ]:


final_all_results["y_pred_final"] = (final_all_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(
    int)
y_pred_all_final = final_all_results["y_pred_final"]

print("Final Results ALL -> ALL (majority voting):")
print(f"Accuracy : {accuracy_score(y_true_all, y_pred_all_final):.4f}")
print(f"Precision: {precision_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_all, y_pred_all_final, labels=[0, 1]))

Final Results ALL -> ALL (majority voting):
Accuracy : 0.5668
Precision: 0.5756
Recall   : 0.5086
F1-score : 0.5400
Confusion matrix:
 [[145  87]
 [114 118]]


In [ ]:
final_acc_gyro_all = evaluate_acc_gyro_only(
    pred_acc_all,
    pred_gyro_all,
    "ALL -> ALL (ACC & GYRO Only)"
)

--- ALL -> ALL (ACC & GYRO Only) ---
Accuracy : 0.5711
Precision: 0.5864
Recall   : 0.4828
F1-score : 0.5296
Confusion matrix:
 [[153  79]
 [120 112]]
